
Adapted from Gemini API Cookbook: https://github.com/google-gemini/cookbook/blob/main/examples/Agents_Function_Calling_Barista_Bot.ipynb, see the original notebook which is under an Apache License.


# Barista Bot: Agents and Function Calling with Google Gemini API

<a href="https://colab.research.google.com/github/jasper-zheng/teaching/blob/main/AIAA_Fall2026/01-FunctionCalling-BaristaBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


This notebook shows a practical example of using automatic function calling with the Gemini API's Python SDK to build an agent. You will define some functions that comprise a café's ordering system, connect them to the Gemini API and write an agent loop that interacts with the user to order café drinks.

To set this notebook to the right environment, click the **Select kernel** button in the top right corner of this notebook, then select **Python Environments...**


Install the google-genai package if you haven't:

In [ ]:
%pip install google-genai

**Get a Google API key:** Go to [https://aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) and hit "Create API key" at the top right. Name it anything you want. Select "Default Gemini Project" as your project. (Don't set up any billing!) Then you'll want to copy your API key (using the copy icon just to the right of the "set up billing" text) and paste it below:

In [ ]:
my_key = "__your_api_key_here__" #Please don't commit any code with your API key to github, or share your key with anyone

In [ ]:
from google import genai

client = genai.Client(api_key=my_key)

## Define functions

To simulate a cafe's order system, create functions that handle the process of a customer's order: adding items, modifying them, confirming and fulfilling the order.

We use two global variables keep track of the process: `order` (the in-progress order) and `placed_order` (the confirmed order sent to the kitchen). Each of the order-editing functions updates the `order`, and once placed, `order` is copied to `placed_order` and cleared.

In the Python SDK you can pass functions directly to the model constructor, where the SDK will inspect the type signatures and docstrings to define the `tools`. For this reason it's important that you correctly type each of the parameters, give the functions sensible names and detailed docstrings.

In [ ]:
from typing import Optional
from random import randint

order = []  # The in-progress order.
placed_order = []  # The confirmed, completed order.


def add_to_order(drink: str, modifiers: Optional[list[str]] = None) -> None:
    """Adds the specified drink to the customer's order, including any modifiers."""
    if modifiers is None:  # Ensures safe handling of None
        modifiers = []
    order.append((drink, modifiers))


def get_order() -> list[tuple[str, list[str]]]:
    """Returns the customer's order."""
    return order


def remove_item(n: int) -> str:
    """Removes the nth (one-based) item from the order.

    Returns:
        The item that was removed.
    """
    item, _ = order.pop(n - 1)
    return item


def clear_order() -> None:
    """Removes all items from the customer's order."""
    order.clear()


def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
        The user's free-text response.
    """
    print("Your order:")
    if not order:
        print("  (no items)")

    for drink, modifiers in order:
        print(f"  {drink}")
        if modifiers:
            print(f'   - {", ".join(modifiers)}')

    return input("Is this correct? ")


def place_order() -> int:
    """Submit the order to the kitchen.

    Returns:
        The estimated number of minutes until the order is ready.
    """
    placed_order[:] = order.copy()
    clear_order()

    # TODO: Implement coffee fulfillment.
    return randint(1, 10)

## Test the functions

With the functions written, test that they work as expected.

**Note:** It's going to pop up a window asking you to proceed, make sure to answer it

In [ ]:
# Test it out!

clear_order()
add_to_order("Latte", ["Extra shot"])
add_to_order("Tea")
remove_item(2)
add_to_order("Tea", ["Earl Grey", "hot"])
confirm_order()

## Define the prompt

Here you define the full Barista-bot prompt. This prompt contains the café's menu items and modifiers and some instructions.

The instructions include guidance on how functions should be called (e.g. "Always `confirm_order` with the user before calling `place_order`"). You can modify this to add your own interaction style to the bot, for example if you wanted to have the bot repeat every request back before adding to the order, you could provide that instruction here.

The end of the prompt includes some jargon the bot might encounter, and instructions of the day - in this case it notes that the cafe has run out of soy milk.

In [ ]:
COFFEE_BOT_PROMPT = """\You are a coffee order taking system and you are restricted to talk only about drinks on the MENU. Do not talk about anything but ordering MENU drinks for the customer, ever.
Your goal is to do place_order after understanding the menu items and any modifiers the customer wants.
Add items to the customer's order with add_to_order, remove specific items with remove_item, and reset the order with clear_order.
To see the contents of the order so far, call get_order (by default this is shown to you, not the user)
Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will display the order items to the user and returns their response to seeing the list. Their response may contain modifications.
Always verify and respond with drink and modifier names from the MENU before adding them to the order.
If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect.
You only have the modifiers listed on the menu below: Milk options, espresso shots, caffeine, sweeteners, special requests.
Once the customer has finished ordering items, confirm_order and then place_order.

Hours: Tues, Wed, Thurs, 10am to 2pm
Prices: All drinks are free.

MENU:
Coffee Drinks:
Espresso
Americano
Cold Brew

Coffee Drinks with Milk:
Latte
Cappuccino
Cortado
Macchiato
Mocha
Flat White

Tea Drinks:
English Breakfast Tea
Green Tea
Earl Grey

Tea Drinks with Milk:
Chai Latte
Matcha Latte
London Fog

Other Drinks:
Steamer
Hot Chocolate

Modifiers:
Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
Espresso shots: Single, Double, Triple, Quadruple; default: Double
Caffeine: Decaf, Regular; default: Regular
Hot-Iced: Hot, Iced; Default: Hot
Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

"dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
"Regular milk" is the same as 'whole milk'.
"Sweetened" means add some regular sugar, not a sweetener.

Soy milk has run out of stock today, so soy is not available.
"""

## Set up the model

In this step you collate the functions into a "system" that is passed as `tools`, instantiate the model and start the chat session.

A retriable `send_message` function is also defined to help with low-quota conversations.

In [ ]:
from google.genai import types

ordering_system = [
    add_to_order,
    get_order,
    remove_item,
    clear_order,
    confirm_order,
    place_order,
]
model_name = "gemini-2.5-flash"

chat = client.chats.create(
    model=model_name,
    config=types.GenerateContentConfig(
        tools=ordering_system,
        system_instruction=COFFEE_BOT_PROMPT,
    ),
)

placed_order = []
order = []

## Chat with Barista Bot

With the model defined and chat created, all that's left is to connect the user input to the model and display the output, in a loop. This loop continues until an order is placed.

Some things to try:
* Ask about the menu (e.g., "what coffee drinks are available?")
* Something on the menu (e.g., "Latte with oat milk")
* Use terms that are not specified in the prompt (e.g., "a strong latte" or "an EB tea")
* Change your mind part way through ("uhh cancel the latte sorry")

In [ ]:
from IPython.display import display, Markdown

print("Welcome to Barista bot! \n\n")

while not placed_order:
    response = chat.send_message(input("> "))
    display(Markdown(response.text))


print("\n\n")
print("[barista bot session over]")
print()
print("Your order:")
print(f"  {placed_order}\n")
print("- Thanks for using Barista Bot!")

## Tasks

**1. Try to break the bot:** Stress-test the barista bot by trying some tricky inputs and note what happens. Some ideas:
- Ask for something not on the menu (e.g. "a babycino" or "orange juice")
- Use ambiguous language (e.g. "a strong one" or "the usual")
- Ask for soy milk (it's out of stock — does the bot remember?)
- Say something completely off-topic (e.g. "what's the weather like?")

**2. Give it a personality:** Modify `COFFEE_BOT_PROMPT` to give the barista bot a distinct character. For example:
- An overly enthusiastic barista who hypes every drink
- A barista who speaks in rhyme
- A grumpy barista who answers in as few words as possible

Capture a short example conversation that shows the personality coming through. How does changing the tone affect usability? Does personality ever get in the way of the task?

**3. Adapt the bot to a new domain:** Pick a different ordering or booking domain and adapt the prompt and functions to match. For instance, a record store (vinyl recommendations + checkout), a plant shop, or a CCI timetable assistant (units, rooms, timetable)

**4. Design an evaluation rubric:** Based on your experiments in task 1, design a simple rubric for evaluating a barista bot. Think about what dimensions matter? (e.g., accuracy, helpfulness, staying on-topic, handling errors gracefully), and how would you score each dimension? (e.g. 1-5 scale, pass/fail)